In [107]:
%load_ext autoreload
%autoreload 2

import sys, os, pathlib
REPO_DIR = pathlib.Path(os.path.dirname(os.path.abspath(os.getcwd()))); print(f"Repo: {str(REPO_DIR)}")
sys.path.insert(0, str(REPO_DIR / "src"))

from tabulate import tabulate
from traffic_flow.types import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Repo: /Users/tristanchaang/MIT/UROP-Traffic/mpc-traffic-flow


[autoreload of traffic_flow.inputs.scenario failed: Traceback (most recent call last):
  File "/Users/tristanchaang/miniconda3/envs/automatic_control_2/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/Users/tristanchaang/miniconda3/envs/automatic_control_2/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/Users/tristanchaang/miniconda3/envs/automatic_control_2/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/Users/tristanchaang/miniconda3/envs/automatic_control_2/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 349, in update_class
    if update_generic(old_obj, new_obj):
  File "/Users/tristanchaang/miniconda3/envs/automatic_control_2/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
  

# A tiny Pyomo control problem

Fill a bucket from **0 to 3 litres in $N$ steps**. At each step, choose an inflow between 0 and 2 litres.

- State: $L_s$ — water already in the bucket. $(0\leq s \leq N)$
- Control: $0 \leq q_t \leq 2$ — water added during step $t$. $(0\leq t < N)$
- Dynamics: $L_{t+1} = L_t + q_t$. $(0 \leq t < N)$
- Constraints: $L_0 = 0, L_N = 3$.
- Objective: $\min \displaystyle\sum_{t=0}^{N-1} q_t^2$.


In [108]:
class BucketModel(PyoModel):
    s: FiniteScalarRangeSet; t: FiniteScalarRangeSet; 
    L: IndexedVar; q: IndexedVar
    dynamics: IndexedConstraint; constraints: ConstraintList; 
    objective: ScalarObjective

N = 6
MAX_INFLOW = 2.0

model = BucketModel()
model.s, model.t = (FiniteScalarRangeSet(0, k-1) for k in [N + 1, N])
model.L, model.q = (IndexedVar(r, within=pyo.NonNegativeReals, bounds=b, initialize=0)
                    for r, b in zip([model.s, model.t], [None, (0, MAX_INFLOW)]))
model.dynamics = IndexedConstraint(model.t, rule=lambda m, t: m.L[t + 1] == m.L[t] + m.q[t])
model.constraints = ConstraintList()
model.constraints.add(model.L[0] == 0)    # Start empty.
model.constraints.add(model.L[N] == 3.0)  # Finish at exactly 3 litres.
model.objective = ScalarObjective(sense = pyo.minimize, expr = sum((t+1) * model.q[t] ** 2 for t in model.t))

status = pyo.SolverFactory("ipopt").solve(model, tee=False, load_solutions=False, options={"max_iter": 100, "tol": 1e-9})

assert status.solver.status in (pyo.SolverStatus.ok, pyo.SolverStatus.warning)
assert status.solver.termination_condition == pyo.TerminationCondition.optimal
assert len(status.solution) > 0

model.solutions.load_from(status) # we didn't load solutions so that we can check success before loading 

rows = [(t, pyo.value(model.L[t]), pyo.value(model.q[t]), pyo.value(model.L[t + 1]))
        for t in model.t]
print(tabulate(rows, floatfmt=".2f", tablefmt="outline", headers=["Step", "Start level", "Inflow", "End level"]))
print(f"Total effort: {pyo.value(model.objective):.2f}")

# model.pprint()


+--------+---------------+----------+-------------+
|   Step |   Start level |   Inflow |   End level |
+========+===============+==========+=============+
|      0 |         -0.00 |     1.22 |        1.22 |
|      1 |          1.22 |     0.61 |        1.84 |
|      2 |          1.84 |     0.41 |        2.24 |
|      3 |          2.24 |     0.31 |        2.55 |
|      4 |          2.55 |     0.24 |        2.80 |
|      5 |          2.80 |     0.20 |        3.00 |
+--------+---------------+----------+-------------+
Total effort: 3.67


# A tiny Pyomo calibration problem

$m=2$ taps fill separate buckets. Their $N$ measured positive levels are known; estimate each tap's **constant flow rate**. Unlike the control example, the unknowns are system parameters, not time-varying controls.

- $0\leq t \leq N$, $0 \leq i < m$.
- Sampling interval $T = 0.5$ seconds.
- Measured levels $\hat L_{t,i}$ in litres. $(0\leq t \leq N$, $0 \leq i < m)$
- Fitted variables: $q_i$ — flow rate of tap $i$ in litres/second.
- Dynamics: $L_{t,i} = L_{t-1,i} + Tq_i$, with $L_{0,i}=\hat L_{0,i}$.
- Objective: $\min \displaystyle\sum_{t=0}^{N}\sum_i (L_{t,i}-\hat L_{t,i})^2$.

Expected fitted rates: **1 and 2 litres/second**, with zero squared error.


In [109]:
class TapCalibrationModel(PyoModel):
    num_taps: int
    t: FiniteScalarRangeSet; i: FiniteScalarRangeSet
    T: ScalarParam; L_hat: IndexedParam
    q: IndexedVar; L: IndexedVar
    dynamics: IndexedConstraint; constraints: ConstraintList
    loss: ScalarObjective

LEVELS = [(0.0, 0.0), (0.5, 1.0), (1.0, 2.0), (1.5, 3.0)]
N = len(LEVELS) - 1

model = TapCalibrationModel()
model.num_taps = len(LEVELS[0])
model.t, model.i = (FiniteScalarRangeSet(0, k) for k in [N, model.num_taps - 1])
model.T = ScalarParam(initialize=0.5)  # Known sampling interval; not fitted.
model.L_hat = IndexedParam(model.t, model.i, initialize={(t, i): LEVELS[t][i] for t in model.t for i in model.i})
model.q = IndexedVar(model.i, within=pyo.NonNegativeReals, bounds=(0, 3), initialize=1)
model.L = IndexedVar(model.t, model.i, within=pyo.NonNegativeReals, initialize=0)
model.constraints = ConstraintList()
for i in model.i: model.constraints.add(model.L[0, i] == model.L_hat[0, i])
model.dynamics = IndexedConstraint(model.t, model.i, 
    expr = {(t, i): pyo.Constraint.Skip if t == 0 else model.L[t, i] == model.L[t - 1, i] + model.T * model.q[i] 
           for t in model.t for i in model.i})
model.loss = ScalarObjective(sense=pyo.minimize,
    expr = sum((model.L[t, i] - model.L_hat[t, i]) ** 2 for t in model.t for i in model.i))

status = pyo.SolverFactory("ipopt").solve(model, tee=False, load_solutions=False, options={"max_iter": 100, "tol": 1e-9})

assert status.solver.status in (pyo.SolverStatus.ok, pyo.SolverStatus.warning)
assert status.solver.termination_condition == pyo.TerminationCondition.optimal
assert len(status.solution) > 0

model.solutions.load_from(status)  # Check success before loading, as in the control example.

rows = [(t, i, pyo.value(model.L_hat[t, i]), pyo.value(model.L[t, i]))
        for t in model.t for i in model.i]
print(tabulate(rows, headers=["Step", "Tap", "Measured level", "Fitted level"],
                floatfmt=".2f", tablefmt="outline"))
print(f"Fitted flow rates: {[pyo.value(model.q[i]) for i in model.i]} litres/second")
print(f"Total squared error: {pyo.value(model.loss):.2f}")

# model.pprint()


+--------+-------+------------------+----------------+
|   Step |   Tap |   Measured level |   Fitted level |
+========+=======+==================+================+
|      0 |     0 |             0.00 |          -0.00 |
|      0 |     1 |             0.00 |          -0.00 |
|      1 |     0 |             0.50 |           0.50 |
|      1 |     1 |             1.00 |           1.00 |
|      2 |     0 |             1.00 |           1.00 |
|      2 |     1 |             2.00 |           2.00 |
|      3 |     0 |             1.50 |           1.50 |
|      3 |     1 |             3.00 |           3.00 |
+--------+-------+------------------+----------------+
Fitted flow rates: [1.000000000045454, 2.0000000000129865] litres/second
Total squared error: 0.00


In [130]:
from traffic_flow.config import CalRef, CalSource
from traffic_flow.results.io import load_some_run
from traffic_flow.results.i24 import run_one_day
from traffic_flow.pipeline import init_state

In [131]:
vsl_path = REPO_DIR / "results_bu" / "i24" / f"i24_11_28" / CalSource.FIXED_RAMPS / "optimal_vsl.npy"
vsl = np.load(vsl_path)
vsl

array([[149.99999999,  70.00000001,  96.02533885, ...,  90.02624539,
        142.25980164, 139.33633265],
       [149.99999999,  70.00000001,  91.88888702, ...,  93.55542621,
        142.26057972, 127.05315265],
       [149.99999999,  70.00000001,  89.96660661, ...,  95.09253928,
        143.26846034, 117.52575765],
       ...,
       [149.99999999,  70.00000001,  75.51857364, ...,  78.86689029,
        137.65683575,  81.71255455],
       [149.99999999,  70.00000001,  75.20466844, ...,  76.97542079,
        132.04580626,  78.59456147],
       [149.99999999,  70.00000001,  75.00001599, ...,  75.00000185,
         86.39157751,  75.00000182]])

In [158]:


run = load_some_run("i24", "11_30", CalRef(CalSource.FIXED_RAMPS, interval=None), study=None)
                        #  where=lambda config: all([config.init_fixed == 150.0]))
spec, traffic = run.scenario.spec, run.scenario.traffic
vsl = run.optimization.vsl
metrics = run_one_day(
    init_state(traffic), traffic.inflow, traffic.downstream_density,
    run.params, dict(enumerate(traffic.lanes)), traffic.lanes,
    traffic.velocity, traffic.density, spec.num_segments, vsl
)
metrics

tts_sim=550.629136840674
tts_opt=383.79906621759255
gt_tt=562.151805104861
metanet_tt=549.259580527159


{'sim_mape': 58.06052227065778,
 'tts_mape': 2.293370662626823,
 'cc': 49.594742913208464,
 'uncontrolled_tts': 550.629136840674,
 'controlled_tts': 383.79906621759255,
 'avg_tt_reduced': 3.1891736192278377,
 'avg_delay_uncontrolled': 6.430467085612156,
 'avg_delay_controlled': 3.2412934663843185}

In [160]:

sim_mape = (abs(traffic.velocity - run.optimization.baseline.velocity[:-1]) / traffic.velocity).mean() * 100

run.optimization.baseline.total_travel_time, run.optimization.controlled.total_travel_time

(550.629136840674, 383.79906621759255)